In [1]:
import os
import re
import fitz
import pandas as pd
from PIL import Image
from rapidocr_onnxruntime import RapidOCR
import numpy as np

In [2]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

pdf_files = [f for f in os.listdir(RAW_DIR) if f.endswith(".pdf")]

print("Jumlah PDF:", len(pdf_files))
print(pdf_files[:5])

Jumlah PDF: 31
['putusan_105_pdt.g_2025_pn_sda_20260624161353.pdf', 'putusan_148_pdt.g_2025_pn_sda_20260624161238.pdf', 'putusan_149_pdt.g_2017_pn_sda_20260624160009.pdf', 'putusan_166_pdt.g_2021_pn_sda_20260624160223.pdf', 'putusan_213_pdt.g_2026_pn_sda_20260624160912.pdf']


In [3]:
ocr = RapidOCR()

def extract_text_ocr(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=200)

        img = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        img_np = np.array(img)

        result, _ = ocr(img_np)

        full_text += f"\n\n--- HALAMAN {page_num} ---\n"

        if result:
            for line in result:
                full_text += line[1] + "\n"

    doc.close()
    return full_text

In [4]:
sample_file = os.path.join(RAW_DIR, pdf_files[0])

text = extract_text_ocr(sample_file)

print(len(text))
print(text[:3000])

11159


--- HALAMAN 1 ---
PENETAPAN
Nomor 105/Pdt.G/2025/PN Sda
esi
DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
Pengadilan Negeri Sidoarjo yang memeriksa dan memutus perkara
perdata pada tingkat pertama, telah memberikan penetapan sebagai berikut
dalam perkara gugatan antara:
1. SRI PUDJIASTUTI, bertempat tinggal di Jalan.Anwar VII/5D-F
Bohar Utara, RT001, RW001, Desa Bohar, Kecamatan
Taman,
Kabupaten
Sidoarjo,
alamat
email
rinata.asfara@gmail.com, sebagai Penggugat I;
2. ASFIRA RACHMAD RINATA, bertempat tinggal di Jalan Raya 51,
RTo01, RW007, Desa Pagentan, Kecamatan Singosari,
Kabupaten
Malang.
alamat
email
rinata.asfara@gmail.com, sebagai Penggugat Il;
3. ASFARA RACHMAD RINATA, bertempat tinggal di Jalan Anwar
VIll/5D-F Bohar Utara, RT001, RW001, Desa Bohar,
Kecamatan Taman, Kabupaten Sidoarjo, alamat email!
rinata.asfara@gmail.com, sebagai Penggugat Ill;
Dalam hal ini Para Penggugat memberikan kuasa Roni Wahyono, S.H.,
M.H., dan kawan-kawan, Para Advokat-Legal Consultant pada

In [5]:
def clean_text(text):
    text = text.lower()

    # hapus disclaimer
    text = re.sub(
        r'disclaimer kepaniteraan.*?ext\.?318',
        ' ',
        text,
        flags=re.DOTALL
    )

    # hapus penanda halaman
    text = re.sub(r'--- halaman \d+ ---', ' ', text)
    text = re.sub(r'halaman \d+', ' ', text)

    # rapikan
    text = re.sub(r'[^a-zA-Z0-9\s./-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(
        r'disclaimer kepaniteraan.*?ext.?318',
        ' ',
        text,
        flags=re.DOTALL
    )

    text = re.sub(r'email\s+kepaniteraan.*?ext.?318', ' ', text)

    return text.strip()

In [6]:
cleaned = clean_text(text)

print(cleaned[:2000])

penetapan nomor 105/pdt.g/2025/pn sda esi demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri sidoarjo yang memeriksa dan memutus perkara perdata pada tingkat pertama telah memberikan penetapan sebagai berikut dalam perkara gugatan antara 1. sri pudjiastuti bertempat tinggal di jalan.anwar vii/5d-f bohar utara rt001 rw001 desa bohar kecamatan taman kabupaten sidoarjo alamat email rinata.asfara gmail.com sebagai penggugat i 2. asfira rachmad rinata bertempat tinggal di jalan raya 51 rto01 rw007 desa pagentan kecamatan singosari kabupaten malang. alamat email rinata.asfara gmail.com sebagai penggugat il 3. asfara rachmad rinata bertempat tinggal di jalan anwar vill/5d-f bohar utara rt001 rw001 desa bohar kecamatan taman kabupaten sidoarjo alamat email rinata.asfara gmail.com sebagai penggugat ill dalam hal ini para penggugat memberikan kuasa roni wahyono s.h. m.h. dan kawan-kawan para advokat-legal consultant pada kantor les hukum/law office roni wahyono s.h. m.h. partner

In [7]:
all_cases = []

for i, file in enumerate(pdf_files, start=1):
    print(f"Memproses {i}/{len(pdf_files)}: {file}")

    path = os.path.join(RAW_DIR, file)
    raw_text = extract_text_ocr(path)
    cleaned_text = clean_text(raw_text)

    all_cases.append({
        "case_id": f"case_{i:03d}",
        "file_name": file,
        "text_full": cleaned_text
    })

df = pd.DataFrame(all_cases)

print(df.shape)
df.head()

Memproses 1/31: putusan_105_pdt.g_2025_pn_sda_20260624161353.pdf
Memproses 2/31: putusan_148_pdt.g_2025_pn_sda_20260624161238.pdf
Memproses 3/31: putusan_149_pdt.g_2017_pn_sda_20260624160009.pdf
Memproses 4/31: putusan_166_pdt.g_2021_pn_sda_20260624160223.pdf
Memproses 5/31: putusan_213_pdt.g_2026_pn_sda_20260624160912.pdf
Memproses 6/31: putusan_21_pdt.g_2026_pn_sda_20260624161015.pdf
Memproses 7/31: putusan_228_pdt.g_2025_pn_sda_20260624161330.pdf
Memproses 8/31: putusan_243_pdt.g_2023_pn_sda_20260624155756.pdf
Memproses 9/31: putusan_252_pdt.g_2025_pn_sda_20260624161145.pdf
Memproses 10/31: putusan_265_pdt.g_2023_pn_sda_20260624160138.pdf
Memproses 11/31: putusan_270_pdt.g_2023_pn_sda_20260624160320.pdf
Memproses 12/31: putusan_290_pdt.g_2025_pn_sda_20260624161319.pdf
Memproses 13/31: putusan_293_pdt.g_2025_pn_sda_20260624161220.pdf
Memproses 14/31: putusan_295_pdt.g_2025_pn_sda_20260624160958.pdf
Memproses 15/31: putusan_300_pdt.g_2022_pn_sda_20260624155956.pdf
Memproses 16/31: put

,case_id,file_name,text_full
0,case_001,putusan_105_pdt.g_2025_pn_sda_20260624161353.pdf,penetapan nomor 105/pdt.g/2025/pn sda esi demi...
1,case_002,putusan_148_pdt.g_2025_pn_sda_20260624161238.pdf,putusan nomor 148/pdt.g/2025/pn sda demi keadi...
2,case_003,putusan_149_pdt.g_2017_pn_sda_20260624160009.pdf,direktori putusan mahkamah agung republik indo...
3,case_004,putusan_166_pdt.g_2021_pn_sda_20260624160223.pdf,direktori putusan mahkamah agung pdt.i.c.3 put...
4,case_005,putusan_213_pdt.g_2026_pn_sda_20260624160912.pdf,penetapan nomor 213/pdt.g/2026/pn sda demi kea...


In [8]:
df.to_csv("../data/processed/cases.csv", index=False)

print("cases.csv berhasil dibuat")

cases.csv berhasil dibuat
